# 🎙️ YAMNet Cough Detection v2 — Transfer Learning + Speech Negatives
# YAMNet 咳嗽检测 v2 — 加入语音负样本

---

## v2 改动 / What's new in v2

**问题 / Problem**: v1 模型 (CoughVid + ESC-50) 把人说话当成咳嗽误判，因为 ESC-50 里完全没有人类语音。
**v1 model misclassified speech as cough — ESC-50 contains no human speech samples.**

**修复 / Fix**: 加入 **LibriSpeech** 语音数据作为额外负样本，让模型学会区分"咳嗽"和"说话"。
**Add LibriSpeech speech clips as additional negatives so model learns speech ≠ cough.**

---

## 数据策略 / Data Strategy

```
正样本 Positive (1000条) ── CoughVid v3 有标签咳嗽音频
                                          ↓
                               合并 2500 条训练数据
                                          ↑
负样本 Negative (1500条)
    ├── ESC-50 人声类 ~280条（喷嚏/呼吸/笑声等）  ← 难负样本
    ├── ESC-50 其他声 720条（环境声）              ← 背景多样性
    └── LibriSpeech 语音 500条 [NEW]              ← 修复说话误判
```

---

## 技术路线 / Pipeline

```
音频文件 → librosa 加载(16kHz单声道) → YAMNet提取1024维Embedding → 全连接分类器 → 咳嗽/非咳嗽
```

---

## 资源路径 / Resource Paths

| 资源 | 路径 |
|------|------|
| YAMNet 模型 | `/kaggle/input/models/google/yamnet/tensorflow2/yamnet/1` |
| CoughVid v3 | `/kaggle/input/datasets/orvile/coughvid-v3/public_dataset_v3/coughvid_20211012` |
| ESC-50 | `/kaggle/input/datasets/mmoreaux/environmental-sound-classification-50` |
| **LibriSpeech [NEW]** | `/kaggle/input/librispeech-clean/LibriSpeech` |
| 输出目录 | `/kaggle/working` |

---

### 🔧 Kaggle 上需要添加的数据集 / Add this dataset on Kaggle

1. 在 Notebook 右侧 **+ Add Input** → **Search Datasets**
2. 搜索 **"librispeech clean"**，选择任一带 `dev-clean` 或 `test-clean` 的数据集
   （推荐 `a24998667/librispeech-clean` 或 `pypiahmad/librispeech-asr-clean`）
3. 添加后注意路径，如不一致请修改下面 `LIBRISPEECH_PATH`


---
## Step 1 · 安装依赖 / Install Dependencies

In [ ]:
!pip install librosa -q
!pip install seaborn -q

---
## Step 2 · 导入库 / Import Libraries

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# 全局随机种子，保证结果可复现
# Global random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'✅ TensorFlow : {tf.__version__}')
print(f'✅ GPU 可用   : {tf.config.list_physical_devices("GPU")}')
print(f'✅ librosa    : {librosa.__version__}')

---
## Step 3 · 全局配置 / Global Configuration

所有关键参数集中管理，需要调整时只改这里。

All parameters managed here — only edit this cell when tuning.

In [ ]:
# ── 路径 / Paths ──────────────────────────────────────────────────────────────
YAMNET_PATH      = '/kaggle/input/models/google/yamnet/tensorflow2/yamnet/1'
COUGHVID_PATH    = '/kaggle/input/datasets/orvile/coughvid-v3/public_dataset_v3/coughvid_20211012'
ESC50_PATH       = '/kaggle/input/datasets/mmoreaux/environmental-sound-classification-50'
ESC50_AUDIO_DIR  = '/kaggle/input/datasets/mmoreaux/environmental-sound-classification-50/audio/audio'
LIBRISPEECH_PATH = '/kaggle/input/datasets/victorling/librispeech-clean/LibriSpeech/dev-clean'
OUTPUT_PATH      = '/kaggle/working'

# ── CoughVid 配置 / CoughVid Config ───────────────────────────────────────────
METADATA_FILE = 'metadata_compiled.csv'
LABEL_COL     = 'status_SSL'
COUGH_LABELS  = ['COVID-19', 'symptomatic', 'healthy']  # 所有有标签的都是真实咳嗽
                                                         # All labeled = real coughs

# ── 样本数量 / Sample Count ───────────────────────────────────────────────────
N_POS         = 1000   # 正样本：咳嗽 / Positive: cough
N_NEG_OTHER   = 720    # ESC-50 非人声 / ESC-50 environmental sounds
N_SPEECH      = 500    # 🆕 LibriSpeech 语音负样本 / NEW: human speech negatives
                       # ESC-50 人声 ~280 + 其他 720 + Speech 500 ≈ 1500 总负样本
                       # ~280 ESC50 human + 720 env + 500 speech = ~1500 negatives

# ── 语音裁剪 / Speech clip filtering ──────────────────────────────────────────
SPEECH_MIN_DUR_S = 0.8   # 跳过过短语音 / skip very short utterances
SPEECH_MAX_DUR_S = 5.0   # 跳过过长语音 / skip very long utterances

# ── ESC-50 人声类别（全部取，约280条）──
HUMAN_CATEGORIES = [
    'sneezing',    # 喷嚏 — 最像咳嗽，最重要的难负样本
    'breathing',   # 呼吸声
    'laughing',    # 笑声
    'crying_baby', # 哭声
    'snoring',     # 打鼾
    'clapping',    # 鼓掌
    'coughing',    # ESC-50 里也有咳嗽，这里作为负样本增加难度
]

# ── 音频配置 / Audio Config ───────────────────────────────────────────────────
SAMPLE_RATE   = 16000  # YAMNet 要求 16kHz
AUDIO_EXTS    = ['.webm', '.ogg', '.wav', '.mp3', '.flac']

# ── 数据集划分 / Dataset Split ────────────────────────────────────────────────
TEST_SIZE     = 0.15
VAL_SIZE      = 0.15

# ── 训练配置 / Training Config ────────────────────────────────────────────────
BATCH_SIZE    = 64
EPOCHS        = 50
LR            = 1e-3
PATIENCE      = 8

print('✅ 配置完成 / Configuration ready')
print(f'   正样本数 / Positive : {N_POS} 条（CoughVid）')
print(f'   负样本数 / Negative : ESC-50 人声~280 + 其他 {N_NEG_OTHER} + LibriSpeech 语音 {N_SPEECH}')


---
## Step 4 · 加载 YAMNet / Load YAMNet

YAMNet 作为**冻结的特征提取器**，权重在训练中不更新，只训练后面的分类层。

YAMNet is a **frozen feature extractor** — weights stay fixed, only the classifier head trains.

In [ ]:
print('⏳ 正在加载 YAMNet... / Loading YAMNet...')
yamnet = tf.saved_model.load(YAMNET_PATH)
print('✅ YAMNet 加载成功 / YAMNet loaded')
print('\n📋 模型签名 / Model signature:')
print(yamnet.signatures)

---
## Step 5 · 定义特征提取函数 / Define Feature Extraction Function

封装音频加载 + YAMNet 推理为一个函数，后续统一调用。

Encapsulate audio loading + YAMNet inference into a reusable function.

In [ ]:
def find_audio_path(uuid_val, audio_dir):
    """
    根据 uuid 在目录中查找对应音频文件
    Find audio file by uuid in directory
    """
    for ext in AUDIO_EXTS:
        p = os.path.join(audio_dir, f'{uuid_val}{ext}')
        if os.path.exists(p):
            return p
    return None


def extract_embedding(audio_path, sr=SAMPLE_RATE):
    """
    从音频文件提取 YAMNet 1024 维 Embedding
    Extract 1024-dim YAMNet embedding from audio file

    Args:
        audio_path : str  音频文件路径 / Path to audio file
        sr         : int  目标采样率 / Target sample rate
    Returns:
        np.ndarray (1024,) 或 None
    """
    try:
        # 1. 加载音频，转成 16kHz 单声道
        #    Load audio, convert to 16kHz mono
        wav, _ = librosa.load(audio_path, sr=sr, mono=True)
        wav = wav.astype(np.float32)

        # 2. 跳过静音文件
        #    Skip silent files
        if np.max(np.abs(wav)) < 1e-6:
            return None

        # 3. 归一化到 [-1.0, 1.0]
        #    Normalize to [-1.0, 1.0]
        wav = wav / np.max(np.abs(wav))

        # 4. YAMNet 推理，取 embeddings
        #    YAMNet inference, get embeddings
        _, embeddings, _ = yamnet(wav)

        # 5. 对时间帧取均值 → 固定 1024 维向量
        #    Average over time frames → fixed 1024-dim vector
        return tf.reduce_mean(embeddings, axis=0).numpy()

    except Exception:
        return None  # 静默跳过失败文件 / Silently skip failed files


print('✅ 特征提取函数定义完成 / Feature extraction functions defined')

---
## Step 6 · 准备数据集 / Prepare Dataset

分三部分组装 2000 条训练数据：
1. **正样本**：CoughVid v3 有标签咳嗽音频，随机取 1000 条
2. **负样本（人声）**：ESC-50 人声类别全取，约 280 条
3. **负样本（其他）**：ESC-50 其他类别随机取 720 条

Assembling 2000 training samples in three parts:
1. **Positives**: 1000 labeled coughs from CoughVid v3
2. **Negatives (human)**: All ~280 human-sound clips from ESC-50
3. **Negatives (other)**: 720 random non-human clips from ESC-50

In [ ]:
# ══════════════════════════════════════════════════════════════
# 第一部分：CoughVid v3 正样本（咳嗽）
# Part 1: CoughVid v3 positive samples (cough)
# ══════════════════════════════════════════════════════════════

print('📂 加载 CoughVid v3 元数据... / Loading CoughVid v3 metadata...')
df_cv = pd.read_csv(os.path.join(COUGHVID_PATH, METADATA_FILE))

# 只保留有 status_SSL 标签的行（所有有标签的都是真实咳嗽录音）
# Keep only labeled rows (all labeled = real cough recordings)
df_cv = df_cv.dropna(subset=[LABEL_COL]).copy()

# 查找对应音频文件
# Find corresponding audio files
df_cv['audio_path'] = df_cv['uuid'].apply(
    lambda uid: find_audio_path(uid, COUGHVID_PATH)
)
df_cv = df_cv[df_cv['audio_path'].notna()].reset_index(drop=True)

print(f'   可用咳嗽样本 / Available cough samples: {len(df_cv)} 条')

# 随机取 N_POS 条
# Randomly sample N_POS
n_pos = min(N_POS, len(df_cv))
df_pos = df_cv.sample(n=n_pos, random_state=SEED)[['audio_path']].copy()
df_pos['binary_label'] = 1
df_pos['source']       = 'coughvid'

print(f'✅ 正样本 / Positive samples: {len(df_pos)} 条咳嗽')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 第二部分：ESC-50 负样本（非咳嗽）
# Part 2: ESC-50 negative samples (non-cough)
# ══════════════════════════════════════════════════════════════

print('📂 加载 ESC-50 元数据... / Loading ESC-50 metadata...')
df_esc = pd.read_csv(os.path.join(ESC50_PATH, 'esc50.csv'))

# 构建完整音频路径
# Build full audio path
df_esc['audio_path'] = df_esc['filename'].apply(
    lambda f: os.path.join(ESC50_AUDIO_DIR, f)
)

# 只保留文件存在的行
# Keep only rows with existing files
df_esc = df_esc[df_esc['audio_path'].apply(os.path.exists)].reset_index(drop=True)
print(f'   ESC-50 总有效样本 / Total valid: {len(df_esc)} 条')

# ── 人声类：全部取（约280条，难负样本）──
# Human categories: take all (~280, hard negatives)
df_human = df_esc[df_esc['category'].isin(HUMAN_CATEGORIES)].copy()
print(f'\n   人声类样本 / Human sound samples: {len(df_human)} 条')
print('   各类分布 / Per-category count:')
for cat, cnt in df_human['category'].value_counts().items():
    print(f'      {cat:<20}: {cnt} 条')

# ── 其他类：随机取 N_NEG_OTHER 条（背景声多样性）──
# Other categories: random N_NEG_OTHER (background diversity)
df_other_pool = df_esc[~df_esc['category'].isin(HUMAN_CATEGORIES)]
n_other = min(N_NEG_OTHER, len(df_other_pool))
df_other = df_other_pool.sample(n=n_other, random_state=SEED).copy()
print(f'\n   其他声音样本 / Other sound samples: {len(df_other)} 条')
print(f'   涵盖类别数   / Categories covered  : {df_other["category"].nunique()} 类')

# ── 合并负样本 ──
# Merge negative samples
df_neg = pd.concat([df_human, df_other]).sample(
    frac=1, random_state=SEED
).reset_index(drop=True)[['audio_path', 'category']].copy()
df_neg['binary_label'] = 0
df_neg['source']       = 'esc50'
df_neg = df_neg.drop(columns=['category'])

print(f'\n✅ 负样本 / Negative samples: {len(df_neg)} 条非咳嗽')

---
## Step 6.5 · 🆕 加载 LibriSpeech 语音负样本 / Load LibriSpeech speech negatives

这是 v2 的核心改动：扫描 LibriSpeech 目录，随机抽取 `N_SPEECH` 段语音作为额外负样本。
LibriSpeech 包含数千个不同说话人的有声读物片段，覆盖男声/女声/各种口音。

The v2 core change: scan LibriSpeech directory, sample `N_SPEECH` speech clips as additional negatives.
LibriSpeech has thousands of audiobook clips from diverse speakers (male/female/accents).

> ⚠️ 如果 `LIBRISPEECH_PATH` 路径不存在，请检查你在 Kaggle 添加的数据集路径并更新 Cell 6
> ⚠️ If `LIBRISPEECH_PATH` doesn't exist, check your Kaggle input path and update Cell 6


In [ ]:
# ══════════════════════════════════════════════════════════════
# 🆕 第二点五部分：LibriSpeech 语音负样本（修复说话误判）
# 🆕 Part 2.5: LibriSpeech speech negatives (fixes speech misclassification)
# ══════════════════════════════════════════════════════════════
import glob

print(f'📂 扫描 LibriSpeech 目录... / Scanning {LIBRISPEECH_PATH}')

if not os.path.exists(LIBRISPEECH_PATH):
    raise FileNotFoundError(
        f'❌ LibriSpeech 路径不存在: {LIBRISPEECH_PATH}\n'
        f'请检查 Kaggle Input 设置，或修改 Cell 6 中的 LIBRISPEECH_PATH'
    )

# 递归扫描所有 .flac / .wav 文件（LibriSpeech 默认是 flac）
# Recursively find all audio files (LibriSpeech uses .flac by default)
speech_files = []
for ext in ['flac', 'wav', 'mp3']:
    speech_files.extend(glob.glob(os.path.join(LIBRISPEECH_PATH, '**', f'*.{ext}'), recursive=True))

print(f'   找到 {len(speech_files)} 段语音文件 / Found {len(speech_files)} audio files')

if len(speech_files) == 0:
    raise RuntimeError(
        '❌ 未找到任何语音文件！请检查 LIBRISPEECH_PATH 是否正确。\n'
        f'   当前路径: {LIBRISPEECH_PATH}\n'
        '   提示：在 Kaggle 上 ls /kaggle/input/ 查看实际数据集名'
    )

# 过滤过短/过长的片段（用音频时长粗筛，不读完整文件）
# Filter by duration (rough check using soundfile, doesn't load whole file)
print(f'   过滤时长 [{SPEECH_MIN_DUR_S}s, {SPEECH_MAX_DUR_S}s] 的片段...')
import soundfile as sf
valid_speech = []
for fp in tqdm(speech_files, desc='Duration check'):
    try:
        info = sf.info(fp)
        dur = info.frames / info.samplerate
        if SPEECH_MIN_DUR_S <= dur <= SPEECH_MAX_DUR_S:
            valid_speech.append(fp)
    except Exception:
        continue

print(f'   过滤后剩余 {len(valid_speech)} 段 / {len(valid_speech)} clips after filtering')

# 随机抽取 N_SPEECH 条
# Randomly sample N_SPEECH
n_speech = min(N_SPEECH, len(valid_speech))
random.shuffle(valid_speech)
speech_paths = valid_speech[:n_speech]

df_speech = pd.DataFrame({
    'audio_path'   : speech_paths,
    'binary_label' : 0,           # not cough
    'source'       : 'librispeech',
})

print(f'\n✅ 语音负样本 / Speech negatives: {len(df_speech)} 条')


In [ ]:
# ══════════════════════════════════════════════════════════════
# 第三部分：合并正负样本，组成最终数据集 (v2: 加入 LibriSpeech)
# Part 3: Merge positive & negative into final dataset (v2: with LibriSpeech)
# ══════════════════════════════════════════════════════════════

# 把 LibriSpeech 也并进负样本
# Merge LibriSpeech into negatives
df_neg_full = pd.concat([df_neg, df_speech], ignore_index=True).sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

df_all = pd.concat([df_pos, df_neg_full]).sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

print('✅ 最终数据集 v2 / Final dataset v2')
print(f'   总计   / Total       : {len(df_all)} 条')
print(f'   正样本 / Positive    : {(df_all.binary_label==1).sum()} 条 (Cough)')
print(f'   负样本 / Negative    : {(df_all.binary_label==0).sum()} 条 (Non-cough)')
print(f'   ├─ ESC-50            : {(df_all.source=="esc50").sum()} 条')
print(f'   └─ LibriSpeech (NEW) : {(df_all.source=="librispeech").sum()} 条')

# 可视化标签分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 正负样本比例
df_all['binary_label'].value_counts().plot(
    kind='bar', ax=axes[0],
    color=['#2ECC71', '#E74C3C'], edgecolor='black'
)
axes[0].set_xticklabels(['Non-Cough (0)', 'Cough (1)'], rotation=0)
axes[0].set_title('正负样本分布 / Label Distribution')
axes[0].set_ylabel('数量 / Count')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(int(bar.get_height())), ha='center')

# 数据来源分布（现在有3个来源）
df_all['source'].value_counts().plot(
    kind='bar', ax=axes[1],
    color=['#3498DB', '#9B59B6', '#F39C12'], edgecolor='black'
)
axes[1].set_title('数据来源 v2 / Data Source v2')
axes[1].set_ylabel('数量 / Count')
axes[1].tick_params(axis='x', rotation=15)
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(int(bar.get_height())), ha='center')

plt.tight_layout()
plt.show()


---
## Step 7 · 提取 YAMNet Embedding / Extract YAMNet Embeddings

遍历所有 2000 条音频，提取 1024 维特征向量。
YAMNet 内部将音频切成 0.96 秒滑动窗口，我们对所有帧取均值得到固定长度向量。

Extract 1024-dim feature vectors from all 2000 audio clips.
YAMNet slices audio into 0.96s windows; we average all frames into one fixed vector.

> ⏱️ 2000 条在 GPU T4 上约 **3–5 分钟** / ~3–5 min on GPU T4

In [ ]:
X_list, y_list, skip_count = [], [], 0

print(f'⏳ 开始提取 {len(df_all)} 条音频的 Embedding...')
print(f'⏳ Extracting embeddings for {len(df_all)} clips...\n')

for _, row in tqdm(df_all.iterrows(), total=len(df_all), desc='Embedding'):
    emb = extract_embedding(row['audio_path'])
    if emb is None:
        skip_count += 1
        continue
    X_list.append(emb)
    y_list.append(row['binary_label'])

X = np.array(X_list, dtype=np.float32)  # (N, 1024)
y = np.array(y_list, dtype=np.float32)  # (N,)

print(f'\n✅ 提取完成 / Extraction done')
print(f'   成功 / Success  : {len(X)}')
print(f'   跳过 / Skipped  : {skip_count}')
print(f'   维度 / X shape  : {X.shape}')
print(f'   咳嗽比例 / Cough%: {y.mean()*100:.1f}%')

# 提取完成后立即保存，下次直接加载跳过此步骤
# Save immediately after extraction — load next time to skip this step
np.savez(os.path.join(OUTPUT_PATH, 'embeddings.npz'), X=X, y=y)
print(f'\n💾 Embedding 已保存 / Saved to: {OUTPUT_PATH}/embeddings.npz')
print('   下次可直接加载，跳过提取步骤 / Next time load directly to skip extraction')

---
## Step 8 · 划分数据集 / Split Dataset

按 **训练 70% / 验证 15% / 测试 15%** 划分，`stratify` 保持各集合正负比例一致。

Split **Train 70% / Val 15% / Test 15%** with stratified sampling.

In [ ]:
# 第一次切分：训练+验证 vs 测试
# First split: train+val vs test
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# 第二次切分：训练 vs 验证
# Second split: train vs val
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=VAL_SIZE, random_state=SEED, stratify=y_tv
)

total = len(X)
print('✅ 数据集划分完成 / Dataset split done')
print(f'   训练集 Train : {len(X_train):>5} ({len(X_train)/total*100:.1f}%)')
print(f'   验证集 Val   : {len(X_val):>5} ({len(X_val)/total*100:.1f}%)')
print(f'   测试集 Test  : {len(X_test):>5} ({len(X_test)/total*100:.1f}%)')

---
## Step 9 · 构建分类器 / Build Classifier

在 YAMNet 的 1024 维 Embedding 后接轻量全连接网络：

```
Input(1024) → Dense(256)+BN+ReLU+Dropout(0.4)
            → Dense(64)+BN+ReLU+Dropout(0.3)
            → Dense(1, sigmoid) → 咳嗽概率
```

In [ ]:
def build_model(input_dim=1024):
    """
    构建咳嗽二分类器
    Build binary cough classifier
    输入 Input : (1024,) YAMNet embedding
    输出 Output: 咳嗽概率 [0,1] / Cough probability
    """
    inputs = tf.keras.Input(shape=(input_dim,), name='yamnet_embedding')

    # 第一个全连接块 / First dense block
    x = tf.keras.layers.Dense(256, name='dense_1')(inputs)
    x = tf.keras.layers.BatchNormalization(name='bn_1')(x)
    x = tf.keras.layers.Activation('relu', name='relu_1')(x)
    x = tf.keras.layers.Dropout(0.4, name='drop_1')(x)

    # 第二个全连接块 / Second dense block
    x = tf.keras.layers.Dense(64, name='dense_2')(x)
    x = tf.keras.layers.BatchNormalization(name='bn_2')(x)
    x = tf.keras.layers.Activation('relu', name='relu_2')(x)
    x = tf.keras.layers.Dropout(0.3, name='drop_2')(x)

    # 输出层：sigmoid 二分类 / Output: sigmoid binary
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='cough_prob')(x)

    return tf.keras.Model(inputs, outputs, name='CoughClassifier')


model = build_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

model.summary()

---
## Step 10 · 训练 / Train

三重回调保障训练质量：

| 回调 | 触发条件 | 作用 |
|------|---------|------|
| `EarlyStopping` | val_auc 连续 8 轮不提升 | 停止训练，恢复最佳权重 |
| `ReduceLROnPlateau` | val_auc 连续 3 轮不提升 | 学习率 × 0.5 |
| `ModelCheckpoint` | 每轮 val_auc 更高时 | 保存最佳模型 |

In [ ]:
best_model_path = os.path.join(OUTPUT_PATH, 'best_cough_classifier_v2.keras')

callbacks = [
    # 早停 / Early stopping
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=PATIENCE,
        restore_best_weights=True, mode='max', verbose=1
    ),
    # 学习率衰减 / LR decay
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5, patience=3,
        min_lr=1e-6, mode='max', verbose=1
    ),
    # 保存最佳模型 / Save best model
    tf.keras.callbacks.ModelCheckpoint(
        filepath=best_model_path,
        monitor='val_auc', save_best_only=True,
        mode='max', verbose=1
    ),
]

print('🚀 开始训练 / Start training...\n')
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)
print('\n✅ 训练完成 / Training complete')

---
## Step 11 · 可视化训练过程 / Visualize Training History

In [ ]:
h = history.history
epochs_ran = range(1, len(h['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('训练曲线 / Training Curves', fontsize=13)

# 准确率 / Accuracy
axes[0].plot(epochs_ran, h['accuracy'],     label='Train', color='#2ECC71')
axes[0].plot(epochs_ran, h['val_accuracy'], label='Val',   color='#E74C3C')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

# AUC
axes[1].plot(epochs_ran, h['auc'],     label='Train', color='#3498DB')
axes[1].plot(epochs_ran, h['val_auc'], label='Val',   color='#E74C3C')
axes[1].set_title('AUC')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

# 损失 / Loss
axes[2].plot(epochs_ran, h['loss'],     label='Train', color='#9B59B6')
axes[2].plot(epochs_ran, h['val_loss'], label='Val',   color='#E74C3C')
axes[2].set_title('Loss')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ 训练曲线已保存 / Curves saved')

---
## Step 12 · 测试集评估 / Test Set Evaluation

在模型从未见过的测试集上评估最终性能。

Final evaluation on the held-out test set.

In [ ]:
# 数值指标 / Numeric metrics
results = model.evaluate(X_test, y_test, verbose=0)
metric_names = ['Loss', 'Accuracy', 'AUC', 'Precision', 'Recall']

print('📊 测试集结果 / Test Set Results')
print('=' * 40)
for name, val in zip(metric_names, results):
    print(f'   {name:<12}: {val:.4f}')

precision = results[3]
recall    = results[4]
f1 = 2 * precision * recall / (precision + recall + 1e-8)
print(f'   {"F1":<12}: {f1:.4f}')

In [ ]:
# 分类报告 + 混淆矩阵 / Classification report + confusion matrix
y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob > 0.5).astype(int)

print('\n📋 分类报告 / Classification Report')
print('=' * 50)
print(classification_report(
    y_test, y_pred,
    target_names=['Non-Cough (0)', 'Cough (1)']
))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Non-Cough', 'Cough'],
    yticklabels=['Non-Cough', 'Cough']
)
ax.set_title('混淆矩阵 / Confusion Matrix')
ax.set_xlabel('预测 / Predicted')
ax.set_ylabel('真实 / True')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Step 13 · 保存模型 / Save Model

In [ ]:
# 保存最终模型（Keras 格式）
# Save final model in Keras format
final_keras = os.path.join(OUTPUT_PATH, 'cough_classifier_final_v2.keras')
model.save(final_keras)
print(f'✅ 模型已保存 / Model saved: {final_keras}')

# 列出所有输出文件
# List all output files
print('\n📁 输出文件 / Output files:')
for f in sorted(os.listdir(OUTPUT_PATH)):
    if not os.path.isdir(os.path.join(OUTPUT_PATH, f)):
        size = os.path.getsize(os.path.join(OUTPUT_PATH, f))
        print(f'   {f:<45} {size/1024:.1f} KB')

---
## Step 14 · 推理示例 / Inference Demo

端到端推理函数：输入任意音频路径 → 输出咳嗽概率和判定结果。

End-to-end inference: input any audio path → cough probability + decision.

In [ ]:
def detect_cough(audio_path, threshold=0.5):
    """
    端到端咳嗽检测 / End-to-end cough detection

    Args:
        audio_path : str   音频文件路径 / Path to audio
        threshold  : float 判定阈值 / Decision threshold (default 0.5)
    Returns:
        dict: probability, is_cough, label, confidence
    """
    emb = extract_embedding(audio_path)
    if emb is None:
        print('❌ 音频处理失败 / Audio processing failed')
        return None

    prob     = float(model.predict(emb[np.newaxis, :], verbose=0)[0][0])
    is_cough = prob > threshold

    result = {
        'probability' : round(prob, 4),
        'is_cough'    : is_cough,
        'label'       : '🔴 咳嗽 COUGH'      if is_cough else '🟢 非咳嗽 NOT COUGH',
        'confidence'  : f'{max(prob, 1-prob)*100:.1f}%'
    }

    print(f'文件       / File       : {os.path.basename(audio_path)}')
    print(f'咳嗽概率   / Probability: {prob:.4f}')
    print(f'判定结果   / Result     : {result["label"]}')
    print(f'置信度     / Confidence : {result["confidence"]}')
    return result


# 用测试集随机一条验证
# Quick demo with a random test sample
print('🔍 推理示例 / Inference Demo')
print('=' * 45)
demo_idx  = 0
demo_prob = float(model.predict(X_test[demo_idx:demo_idx+1], verbose=0)[0][0])
demo_pred = '🔴 Cough' if demo_prob > 0.5 else '🟢 Non-Cough'
demo_true = '🔴 Cough' if y_test[demo_idx] == 1 else '🟢 Non-Cough'
print(f'真实标签 / True  : {demo_true}')
print(f'预测结果 / Pred  : {demo_pred} (prob={demo_prob:.4f})')
print(f'\n📌 使用方法 / Usage:')
print("   detect_cough('/path/to/audio.wav')")

---
## 总结 / Summary (v2)

| 步骤 | 内容 |
|------|------|
| 1–2 | 安装依赖，导入库 |
| 3   | 全局配置（含 LIBRISPEECH_PATH）|
| 4   | 加载 YAMNet |
| 5   | 定义特征提取函数 |
| 6   | 组装数据集 (Part 1: CoughVid → Part 2: ESC-50 → **Part 2.5: LibriSpeech 🆕** → Part 3: Merge) |
| 7   | 提取 1024 维 Embedding |
| 8   | 划分训练/验证/测试集 70/15/15 |
| 9   | 构建分类器 |
| 10  | 训练（早停 + LR衰减 + 最佳模型保存）|
| 11  | 可视化训练曲线 |
| 12  | 测试集评估 + 混淆矩阵 |
| 13  | 保存模型 → `best_cough_classifier_v2.keras` |
| 14  | 推理示例 |

---

## v2 vs v1 关键差异 / Key Differences

| | v1 (原版) | v2 (当前) |
|--|--|--|
| 正样本 | CoughVid × 1000 | CoughVid × 1000 |
| 负样本 | ESC-50 × 1000 | ESC-50 × 1000 + **LibriSpeech 语音 × 500** |
| 总数 | 2000 | ~2500 |
| 解决问题 | — | ✅ 修复"说 hello 被判咳嗽"的 bug |

---

## 部署到 FluGuard 后端 / Deploy to FluGuard backend

训练完成后下载 `best_cough_classifier_v2.keras`，重命名为 `best_cough_classifier.keras`，
放到本地 `cough-detector/` 目录，重启后端即可。

After training, download `best_cough_classifier_v2.keras`, rename to `best_cough_classifier.keras`,
drop into local `cough-detector/`, restart backend.
